# CityFlow Traffic Simulation
### فقط از منوی بالا **Runtime ← Run all** بزنید
---

In [ ]:
# ── مرحله ۱: همگام Git + نصب افزونه (فقط در صورت نیاز) ─────────────
import os, subprocess, hashlib, glob

REPO = '/content/repo'
BRANCH = 'main'

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

def glob_cityflow_so():
    return glob.glob(f'{REPO}/cityflow.cpython-*.so')

def native_build_fingerprint(repo):
    h = hashlib.sha256()
    paths = [f'{repo}/CMakeLists.txt', f'{repo}/install.sh']
    paths += sorted(glob.glob(f'{repo}/src/**/*.cpp', recursive=True))
    paths += sorted(glob.glob(f'{repo}/src/**/CMakeLists.txt', recursive=True))
    for p in paths:
        try:
            with open(p, 'rb') as f:
                h.update(os.path.basename(p).encode())
                h.update(f.read())
        except OSError:
            continue
    return h.hexdigest()

# برای حذف کامل /content/repo و شروع از صفر این را True کنید
FORCE_FULL_REINSTALL = False

if FORCE_FULL_REINSTALL:
    sh(f'rm -rf "{REPO}"')

if not os.path.isdir(os.path.join(REPO, '.git')):
    print('⏳ کلون مخزن...')
    sh(f'git clone https://github.com/persiagfx/cityflow.git "{REPO}" --quiet')
else:
    print('⏳ همگام‌سازی با GitHub (reset به آخرین origin/%s)...' % BRANCH)
    sh(f'git -C "{REPO}" fetch origin')
    sh(f'git -C "{REPO}" checkout {BRANCH}')
    sh(f'git -C "{REPO}" reset --hard origin/{BRANCH}')

fp = native_build_fingerprint(REPO)
marker = os.path.join(REPO, '.colab_native_fingerprint')
so_ok = len(glob_cityflow_so()) > 0

need_build = True
if so_ok and os.path.isfile(marker):
    try:
        with open(marker) as mf:
            if mf.read().strip() == fp:
                need_build = False
    except OSError:
        pass

if need_build:
    print('⏳ کامپایل CityFlow (~۳–۴ دقیقه؛ اولین بار یا بعد از تغییر C++/CMake)...')
    sh(f'bash "{REPO}/install.sh"')
    with open(marker, 'w') as mf:
        mf.write(fp)
else:
    print('✓ افزونهٔ Python همین نسخه قبلاً build شده؛ frontend با Git به‌روز شد.')

print('✓ مرحلهٔ ۱ تمام')

In [ ]:
# ── مرحله ۲: اجرای شبیه‌سازی ──────────────────────────
import sys, os, shutil

sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')

import cityflow

STEPS = 300  # تعداد گام‌های شبیه‌سازی

eng = cityflow.Engine('examples/config.json', thread_num=4)
for i in range(STEPS):
    eng.next_step()

shutil.copy('examples/replay.txt',          'frontend/replay.txt')
shutil.copy('examples/replay_roadnet.json', 'frontend/replay_roadnet.json')

print(f'✓ Simulation done — {STEPS} steps | vehicles: {eng.get_vehicle_count()}')

In [ ]:
# ── مرحله ۳: نمایش (تونل عمومی — در صورت None لاگ پایین را بخوانید) ─────
import subprocess, threading, time, re, os, socket

PORT = 8765
HOST = '127.0.0.1'
LOG = '/tmp/cityflow_cf.log'
URL_RE = re.compile(r'https://[a-zA-Z0-9_.-]+\.trycloudflare\.com')

def strip_ansi(b):
    return re.sub(rb'\x1b\[[0-9;]*[a-zA-Z]', b'', b).decode('utf-8', errors='ignore')

def wait_http_port(secs=30):
    deadline = time.time() + secs
    while time.time() < deadline:
        try:
            s = socket.create_connection((HOST, PORT), timeout=2)
            s.close()
            return True
        except OSError:
            time.sleep(0.4)
    return False

def run_server():
    subprocess.run(
        [
            'python3', '-m', 'http.server', str(PORT),
            '--bind', HOST,
            '--directory', '/content/repo/frontend',
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
time.sleep(0.5)

threading.Thread(target=run_server, daemon=True).start()
print('⏳ منتظر بالا آمدن سرور محلی…')
if not wait_http_port(45):
    print('✗ سرور روی پورت', PORT, 'بالا نیامد — یک بار Runtime → Restart runtime و دوباره از سلول ۳.')
    raise SystemExit(1)
print('✓ سرور محلی آماده است')

bin_path = '/content/cloudflared'
if not os.path.isfile(bin_path):
    print('⏳ دانلود cloudflared…')
    r = subprocess.run(
        [
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', bin_path,
        ],
    )
    if r.returncode != 0 or not os.path.isfile(bin_path):
        print('✗ wget cloudflared ناموفق — اینترنت کولب را چک کنید.')
        raise SystemExit(1)
    os.chmod(bin_path, 0o755)

subprocess.run([bin_path, '--version'], capture_output=True)

url = None
for attempt in range(1, 4):
    print(f'⏳ ساخت لینک TryCloudflare (تلاش {attempt}/۳، تا ~۱۲۰ ثانیه)…')
    subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
    time.sleep(1.2)
    try:
        os.remove(LOG)
    except OSError:
        pass
    logf = open(LOG, 'wb', buffering=0)
    proc = subprocess.Popen(
        [
            bin_path, 'tunnel', '--no-autoupdate',
            '--url', f'http://{HOST}:{PORT}',
        ],
        stdout=logf,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
    )
    for _ in range(120):
        time.sleep(1)
        try:
            with open(LOG, 'rb') as rf:
                blob = rf.read()
            m = URL_RE.search(strip_ansi(blob))
            if m:
                url = m.group(0).rstrip('.,;)')
                break
        except OSError:
            pass
        if proc.poll() is not None:
            print('  (cloudflared زود تمام شد — تلاش بعدی)')
            break
    try:
        logf.close()
    except Exception:
        pass
    if url:
        break

print()
print('=' * 55)
print(f'  🌐 {url}')
print('=' * 55)
print()
if url is None:
    print('  ⚠ لینک نیامد.')
    print('  • حتماً نوت‌بوک را از GitHub بروز کنید:')
    print('    https://github.com/persiagfx/cityflow/blob/main/CityFlow_Colab.ipynb')
    print('  • Runtime → Restart runtime ، بعد فقط سلول ۱ تا ۳ را دوباره اجرا کنید.')
    print()
    print('  ── آخرین خروجی cloudflared (برای دیباگ) ──')
    try:
        tail = open(LOG, 'rb').read()[-3500:]
        print(strip_ansi(tail) or '(خالی)')
    except OSError as e:
        print('(لاگ خوانده نشد)', e)
else:
    print('  ① لینک را باز کنید — Roadnet: replay_roadnet.json ، Replay: replay.txt ، Start')
    print()
    print('  ⚠ اگر Error ۱۰۳۳ دیدید: همین سلول را دوباره Run کنید (لینک موقت است).')
print()
print('  💡 تا وقتی Runtime زنده است، سرور و تونل در پس‌زمینه می‌مانند.')
